In [1]:
import pandas as pd
import re
from ipynb.fs.defs.functions import validate_df

# XLS-Datei einlesen
# Quelle: https://www.kbv.de/infothek/zahlen-und-fakten/gesundheitsdaten/aerzte-regionale-verteilung
df = pd.read_excel("../data/raw/I.1.1.8.xls", skiprows=3, sheet_name="2024")
# Außerdem nötig: hilfe_2024.csv


# alle Spalten finden, die sich auf Kinderärzte beziehen
cols_kinder = df.columns[
    df.columns.get_level_values(0)
    .str.contains("kind", case=False, na=False)
]

# Überblick:
print(df.info())
print(cols_kinder)

WARNING *** file size (5976146) not 512 + multiple of sector size (512)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Columns: 129 entries, Regionsschlüssel to Gesonderte Fachärztliche Versorgung und übrige Arztgruppen.6
dtypes: float64(1), object(128)
memory usage: 518.1+ KB
None
Index(['Kinderärzte', 'Kinderärzte.1', 'Kinderärzte.2', 'Kinderärzte.3',
       'Kinderärzte.4', 'Kinderärzte.5', 'Kinderärzte.6',
       'Kind.Jug.Psychiater', 'Kind.Jug.Psychiater.1', 'Kind.Jug.Psychiater.2',
       'Kind.Jug.Psychiater.3', 'Kind.Jug.Psychiater.4',
       'Kind.Jug.Psychiater.5', 'Kind.Jug.Psychiater.6'],
      dtype='object')


In [2]:
#relevante Spalten filtern
df = df[
    [
        "Name",
        "Regionstyp",
        'Kinderärzte.6',
        'Kind.Jug.Psychiater.6'
    ]
]

df = df.rename(columns={'Kinderärzte.6': 'Kinderarztdichte','Kind.Jug.Psychiater.6': 'KJP-Dichte'})
print(df)

                     Name Regionstyp                  Kinderarztdichte  \
0                     NaN        NaN  Arztdichte (Ärzte je 100.000 EW)   
1      Schleswig-Holstein  KV-Region                               9.7   
2                 Hamburg  KV-Region                              12.8   
3                  Bremen  KV-Region                              14.5   
4           Niedersachsen  KV-Region                               9.5   
..                    ...        ...                               ...   
509   Saalfeld-Rudolstadt     Kreise                               6.9   
510  Saale-Holzland-Kreis     Kreise                                12   
511      Saale-Orla-Kreis     Kreise                               6.3   
512                 Greiz     Kreise                               7.3   
513      Altenburger Land     Kreise                              13.5   

                           KJP-Dichte  
0    Arztdichte (Ärzte je 100.000 EW)  
1                              

In [3]:
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")

print(df["Regionstyp"].unique())
print(df.groupby("Regionstyp")["KJP-Dichte"].apply(lambda s: s.isna().sum()))
print(df.groupby("Regionstyp")["Kinderarztdichte"].apply(lambda s: s.isna().sum()))

[nan 'KV-Region' 'Raumordnungsregionen' 'Kreise']
Regionstyp
KV-Region                 0
Kreise                  400
Raumordnungsregionen      0
Name: KJP-Dichte, dtype: int64
Regionstyp
KV-Region               0
Kreise                  0
Raumordnungsregionen    0
Name: Kinderarztdichte, dtype: int64


In [4]:
# da KJPs nur für Raumordnungsregionen ausgewertet wurden, nicht für Kreise, werden 2 Df's erstellt, einer für die KJPs aggregriert auf Raumordnungsregionen, einer für Kinderärzte in Kreisen/kreisfreien Städten
df_ka = df.copy()

df_ka = df_ka.loc[df["Regionstyp"].astype(str).str.contains("Kreis", case=False, na=False)]
df_ka = df_ka.drop(columns=["Regionstyp"])
df_ka = df_ka.drop(columns=["KJP-Dichte"])

df_kjp = df.copy()

df_kjp = df_kjp.loc[df["Regionstyp"].astype(str).str.contains("Raumordnungsregion", case=False, na=False)]
df_kjp = df_kjp.drop(columns=["Regionstyp"])
df_kjp = df_kjp.drop(columns=["Kinderarztdichte"])
print(df_ka)
print(df_kjp)

                     Name Kinderarztdichte
114      Flensburg, Stadt               13
115           Kiel, Stadt             16.6
116         Lübeck, Stadt             12.4
117     Neumünster, Stadt              8.8
118          Dithmarschen              7.4
..                    ...              ...
509   Saalfeld-Rudolstadt              6.9
510  Saale-Holzland-Kreis               12
511      Saale-Orla-Kreis              6.3
512                 Greiz              7.3
513      Altenburger Land             13.5

[400 rows x 2 columns]
                            Name KJP-Dichte
18      Schleswig-Holstein Mitte          3
19       Schleswig-Holstein Nord        1.1
20        Schleswig-Holstein Ost        2.4
21        Schleswig-Holstein Süd        1.4
22   Schleswig-Holstein Süd-West        0.7
..                           ...        ...
109                    Magdeburg        1.2
110              Mittelthüringen        1.2
111                Nordthüringen        0.6
112                 

In [5]:
# Filtern der Kreise/Städte NRW für die Kinderarztdichte
df2 = pd.read_csv("../data/processed/hilfen_2024.csv")

names = (
    df2["Name"]
    .dropna()
    .astype(str)
    .str.lower()
    .apply(re.escape)  
)

pattern = r"\b(" + "|".join(names) + r")\b"

df_ka = df_ka.loc[
    df["Name"]
      .astype(str)
      .str.lower()
      .str.contains(pattern, na=False, regex=True)
]

df_ka["Name"] = df_ka["Name"].str.replace(", Stadt", "", regex=False).str.strip()
mask = df_ka["Name"].astype(str).str.contains(r"\baachen\b", case=False, na=False)
df_ka.loc[mask, "Name"] = "Aachen"
print(df_ka.info())

<class 'pandas.core.frame.DataFrame'>
Index: 53 entries, 177 to 229
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Name              53 non-null     object
 1   Kinderarztdichte  53 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB
None


C:\Users\inaba\AppData\Local\Temp\ipykernel_11832\528826223.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, na=False, regex=True)


In [6]:
df_ka["Kinderarztdichte"] = pd.to_numeric(df["Kinderarztdichte"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
df_kjp["KJP-Dichte"] = pd.to_numeric(df["KJP-Dichte"].astype(str).str.replace(",", ".", regex=False), errors="coerce")

In [7]:
# Tabelle mit Raumordnungsregionen einlesen
df_ror = pd.read_csv("../data/processed/ror.csv")
df_kjp = df_kjp.rename(columns={"Name":"ROR"})  

# mit KJP-Dichte-Tabelle matchen, um Kreise/Städte den Raumordnungsregionen zuzuorten
df_kjp = df_kjp.rename(columns={"Name":"ROR"}) 
df_merged = df_ror.merge(df_kjp, on=["ROR"], how="left")
print(df_merged.isna().any().any())


False


In [8]:
# prüfen, ob Namen alle übereinstimmen
set_merged = set(df_merged["Name"])
set_ka = set(df_ka["Name"])
no_match = set_merged.symmetric_difference(set_ka)
print(no_match)

set()


In [11]:
df = df_ka.merge(df_merged, on=["Name"], how="left")


validate_df(
    df,
    not_null=["Kinderarztdichte", "KJP-Dichte"],
    numeric=["Kinderarztdichte", "KJP-Dichte"],
    bounds={"Kinderarztdichte": (0, 100), "KJP-Dichte": (0, 100)},
    key_cols=["Name"]
)

DataFrame: alle Checks bestanden


[]

In [12]:
df.to_csv("../data/processed/arztdichte_2024.csv", index=False)